## Multiple Linear Regression - Predicting Health Status from Demographic Data

In this assignment, you will analyze data from the [Behavioral Risk Factor Surveillance System (BRFSS) data](https://www.cdc.gov/brfss/annual_data/annual_data.htm). The objective is to construct a self-contained Jupyter Notebook that predicts health outcomes using income, body mass index (BMI), and education as explanatory variables. Importantly, your solution should be designed to handle the dataset without loading all data into memory simultaneously.

Before beginning, carefully
1. Review the Lecture Slides on Multiple Linear Regression, specifically:
   - Slides 27–31: Derivation of the formula for multiple linear regression coefficients $\hat{\beta} = (X^TX)^{-1}X^Ty$.
   - Slides 36–58: A step-by-step algorithm for computing the regression coefficients and the adjusted R-squared statistic in a single pass over the data.
     ```mermaid
        graph TB;
        DMS["Dataset Metadata Setup<br>Steps 1-4"]
        EF["Extract Fields<br>Steps 5-9"]
        RNV["Remove Null Values<br>Step 10"]
        CSMV["Compute Statistics Matrix A and Vector z<br>Steps 11-12"]
        CRPV["Compute Regression Parameter Vector β<br>Step 13"]
        EM["Compute R^2_adj<br>Steps 14-16"]
        RI["Results Interpretation<br>Steps 17-18"]

        DMS --> EF
        EF --> RNV
        RNV --> CSMV
        CSMV --> CRPV
        CRPV --> EM
        EM --> RI

        subgraph "Data Extraction"
            DMS
            EF
            RNV
        end

        subgraph "Compute Linear Regression"
            CSMV
            CRPV
        end

        subgraph "Model Evaluation"
            EM
            RI
        end
        ```

2. Review the Provided Script Template.
   - Examine the sample Python script template in [`linearRegression.zip`](https://rutgersconnect-my.sharepoint.com/:f:/g/personal/hz333_connect_rutgers_edu/Eu7WWRxo7hZLiPlugT-tfucBSKVD3lc4gRxF3xZNlg9emg?e=e5fap3)
   - Download the BRFSS data files for years 2011, 2012, 2013, and 2014 into the `./data` directory. Do not unzip the downloaded files.
   - Complete the missing sections of the script. You are required to implement a total of **four lines** of code.
   - Execute the completed script to compute the regression coefficients and the adjusted R-squared statistic by running the following command in your terminal:
     ```bash
     python -m ScalableAlgorithms.PythonScripts.linearRegression
     ```

**Your task**:
- Adapt the provided Python script template to implement a function `regression_health` with the following specifications:
  - INPUT:
    - `data_dir`: Path to the directory containing the BRFSS data files (the zip files should remain compressed).
    - `columns`: A non-empty list of column names to be used as predictors (independent variables).
  - OUTPUT:
    - `beta`: A NumPy array of regression coefficients. The first element corresponds to the intercept $\hat{\beta}_0$, followed by the coefficients for each predictor in the order they are listed in `columns`.
    - `adj_r2`: The adjusted R-squared statistic of the regression model.

### Tool Functions

In [ ]:
# STEP 1 START
import sys
import os
import importlib
import pathlib
import numpy as np
import zipfile
from io import TextIOWrapper
# STEP 1 END

In [ ]:
def fieldDictBuild():
    fieldDict = dict.fromkeys([0, 1, 2, 3, 11, 12, 13, 14])

    fieldDict[11] = {
        "genhlth": 73,
        "bmi": (1533, 1536),
        "income": (124, 125),
        "education": 122,
    }
    fieldDict[12] = {
        "genhlth": 73,
        "bmi": (1644, 1647),
        "income": (116, 117),
        "education": 114,
    }
    fieldDict[13] = {
        "genhlth": 80,
        "bmi": (2192, 2195),
        "income": (152, 153),
        "education": 150,
    }
    fieldDict[14] = {
        "genhlth": 80,
        "bmi": (2247, 2250),
        "income": (152, 153),
        "education": 150,
    }

    return fieldDict

def getIncome(incomeString):
    if incomeString != "  ":
        income = int(incomeString)
    else:
        income = 9
    return income

def convertBMI(bmiString, shortYear):
    bmi = 0
    if shortYear == 0 and bmiString != "999":
        bmi = 0.1 * float(bmiString)
    if shortYear == 1 and bmiString != "999999":
        bmi = 0.0001 * float(bmiString)
    if 2 <= shortYear <= 10 and bmiString != "9999":
        bmi = 0.01 * float(bmiString)
    if shortYear > 10 and bmiString != "    ":
        bmi = 0.01 * float(bmiString)
    return bmi

def getEducation(educationString):
    if educationString != " ":
        education = int(educationString)
    else:
        education = 9
    return education

def getHlth(hlthString):
    if hlthString != " ":
        genhlth = int(hlthString)
        if genhlth > 6:
            genhlth = -1
    else:
        genhlth = -1

    assert genhlth in (-1, 1, 2, 3, 4, 5, 6)
    return genhlth


### Main Regression Function

In [ ]:
def regression_health(data_dir, columns):
    """
    Perform linear regression on health data.
    IN: data_dir, str, directory containing data files
    IN: columns, list of str, column names to use as features
    OUT: beta, np.array of shape (len(columns) + 1,), regression coefficients
    OUT: adj_r2, float, adjusted R-squared of the regression
    """

    # Adapt linearRegression.py here
    # Function call: regression_health('./data', ['education', 'income', 'bmi'])
    # Step 1: Import necessary modules (Achieved above)
    # Step 2: Initalize fieldDictBuild() (Achieved above)

    # STEP 3 START
    fieldDict = fieldDictBuild()
    # STEP 3 END
    # STEP 11 START
    q = len(columns) + 1
    A = np.zeros(shape=(q, q))
    z = np.matrix(np.zeros(shape=(q, 1)))
    sumSqrs = 0
    n = 0
    # STEP 11 END
    # STEP 4: iterate over files
    for filename in sorted(os.listdir(data_dir)):
        try:
            shortYear = int(filename[6:8])            # Yeras 11, 12, 13, 14 in the 21st century
        except ValueError:
            continue

        fields = fieldDict.get(shortYear)
        if fields is None or shortYear < 11:
            continue

        # STEP 5 START: These are the field positions for each year
        fEduc = fields["education"]
        sInc, eInc = fields["income"]
        sBMI, eBMI = fields["bmi"]
        fGH = fields["genhlth"]
        minLen = max(fEduc, eInc, eBMI, fGH)
        # STEP 5 END

        # STEP 6 START: Go through ZIP files for each year
        with zipfile.ZipFile(os.path.join(data_dir, filename)) as zf:
            names = zf.namelist()
            assert len(names) == 1
            with zf.open(names[0]) as f:
                for record in TextIOWrapper(f, encoding="latin-1"):
                    if len(record) < minLen:          # blank/short trailing line
                        continue

                    education = getEducation(record[fEduc - 1])
                    income = getIncome(record[sInc - 1:eInc])
                    bmi = convertBMI(record[sBMI - 1:eBMI], shortYear)
                    # STEP 6 END

                    # STEP 7 EDITED IN functions.py
                    # STEP 8 EDITED IN functions.py

                    # STEP 9 START
                    y = getHlth(record[fGH - 1])
                    # STEP 9 END

                    # STEP 10 START: drop missing / refused / invalid
                    try:
                        if education < 9 and income < 9 and 0 < bmi < 99 and y != -1:
                            values = {"education": education, "income": income, "bmi": bmi}
                            x = np.matrix([1.0] + [values[c] for c in columns]).T    # q x 1 column
                        else:
                            1 / 0
                        # STEP 12 START
                        A += x * x.T # 4 x 4 matrix for X^T X
                        z += x * y # 4 x 1 vector for X^T Y
                        sumSqrs += y**2 # Y^T Y
                        n += 1 # row counter
                        # STEP 12 END
                    except ZeroDivisionError:
                        pass
                    # STEP 10 END



    # STEP 4 END
    # STEP 13 START
    beta = np.asarray(np.linalg.inv(A) * z).flatten()  # β = (X^T X)^(-1) X^T y
    # STEP 13 END

    ybar = z[0,0] / n
    varEst = (sumSqrs - (np.matrix(beta)*z)[0,0]) / (n-q)
    adj_r2 = float(1-varEst / ((sumSqrs - n * ybar**2) / (n-1)))
    
    return beta, adj_r2